impoting libraries

In [6]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

loading dataset


In [10]:
housing = fetch_california_housing()

df = pd.DataFrame(housing.data, columns=housing.feature_names)

df['Price'] = housing.target

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
print(df.head())

Dataset Shape: (20640, 9)

First 5 Rows:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  Price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  


Defining features and target

In [11]:
X = df.drop('Price', axis=1)

y = df['Price']

splitting dataset

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


Training Data Shape: (16512, 8)
Testing Data Shape: (4128, 8)


linear regression model



In [13]:
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

# Predictions
lr_predictions = lr_model.predict(X_test)

# Evaluation
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print("\n========== LINEAR REGRESSION ==========")

print("RMSE :", lr_rmse)
print("R2 Score :", lr_r2)



========== LINEAR REGRESSION ==========
RMSE : 0.7455813830127764
R2 Score : 0.5757877060324508


decision tree model


In [14]:
dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train, y_train)

predictions = dt_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))

r2 = r2_score(y_test, predictions)

print("RMSE:", rmse)

print("R2 Score:", r2)

RMSE: 0.7037294974840077
R2 Score: 0.622075845135081


Predictions and Evaluation

In [15]:
# Predictions

dt_predictions = dt_model.predict(X_test)

# RMSE

dt_rmse = np.sqrt(
    mean_squared_error(y_test, dt_predictions)
)

# R2 Score

dt_r2 = r2_score(y_test, dt_predictions)

print("========== DECISION TREE RESULTS ==========")

print("RMSE :", dt_rmse)

print("R2 Score :", dt_r2)

========== DECISION TREE RESULTS ==========
RMSE : 0.7037294974840077
R2 Score : 0.622075845135081


Overfitting Check

In [16]:
train_score = dt_model.score(X_train, y_train)
test_score = dt_model.score(X_test, y_test)

print("========== OVERFITTING CHECK ==========")

print("Training Score :", train_score)

print("Testing Score :", test_score)

if train_score > test_score:
    print("\nModel may be OVERFITTING")
else:
    print("\nModel is performing well")

========== OVERFITTING CHECK ==========
Training Score : 1.0
Testing Score : 0.622075845135081

Model may be OVERFITTING


Cross Validation

In [17]:
cv_scores = cross_val_score(
    dt_model,
    X,
    y,
    cv=5,
    scoring='r2'
)

print("========== CROSS VALIDATION ==========")

print("Cross Validation Scores:")

print(cv_scores)

print("\nAverage CV Score:")

print(cv_scores.mean())

========== CROSS VALIDATION ==========
Cross Validation Scores:
[0.27093461 0.41372445 0.43912441 0.23566991 0.41875969]

Average CV Score:
0.355642615410327


Hyperparameter Tuning

In [18]:
param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("========== BEST PARAMETERS ==========")

print(grid_search.best_params_)

========== BEST PARAMETERS ==========
{'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}


Best Tuned Model

In [19]:
best_model = grid_search.best_estimator_

best_predictions = best_model.predict(X_test)

best_rmse = np.sqrt(
    mean_squared_error(y_test, best_predictions)
)

best_r2 = r2_score(
    y_test,
    best_predictions
)

print("========== TUNED MODEL RESULTS ==========")

print("RMSE :", best_rmse)

print("R2 Score :", best_r2)

========== TUNED MODEL RESULTS ==========
RMSE : 0.6390654005312799
R2 Score : 0.6883380738855668


Model Comparison

In [20]:
comparison = pd.DataFrame({
    'Model': [
        'Decision Tree Before Tuning',
        'Decision Tree After Tuning'
    ],
    'RMSE': [
        dt_rmse,
        best_rmse
    ],
    'R2 Score': [
        dt_r2,
        best_r2
    ]
})

print("========== MODEL COMPARISON ==========")

print(comparison)

========== MODEL COMPARISON ==========
                         Model      RMSE  R2 Score
0  Decision Tree Before Tuning  0.703729  0.622076
1   Decision Tree After Tuning  0.639065  0.688338
